# Chapter 5. EDA (탐색적 데이터 분석)

탐색적 데이터 분석(EDA)은 데이터 분석 전, 데이터의 특성·구조·품질을 파악하는 과정입니다.

| 단계 | 내용 |
|------|------|
| 3-1. 기본 정보 확인 | shape, head, info, describe, dtype, unique |
| 3-2. 대표값 분석 | 평균(mean), 중앙값(median), 최빈값(mode) |
| 3-3. 결측치·이상치 처리 | isnull, dropna, fillna / IQR, Z-score |
| 3-4. 산포도 분석 | 범위(range), 분산(var), 표준편차(std), IQR |
| 3-5. 분포 파악 | 사분위수(Q1/Q2/Q3), 박스플롯 |
| 3-6. 데이터 변형 | Pivot(Long→Wide), Melt(Wide→Long) |

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

mpg = sns.load_dataset('mpg')
penguins = sns.load_dataset('penguins')
print('mpg shape:', mpg.shape)
print('penguins shape:', penguins.shape)

## 3-1. 기본 정보 확인

데이터를 처음 받았을 때 **가장 먼저 실행해야 할 함수들**입니다.

| 함수 | 용도 | 주요 출력 |
|------|------|----------|
| `df.shape` | 행·열 수 확인 | (행수, 열수) tuple |
| `df.head(n)` | 앞 n행 출력 | 기본값 n=5 |
| `df.tail(n)` | 뒤 n행 출력 | 기본값 n=5 |
| `df.info()` | 타입·결측치 요약 | dtype, non-null count |
| `df.describe()` | 수치형 통계 요약 | count, mean, std, min, 25%, 50%, 75%, max |
| `df.dtypes` | 각 열의 데이터 타입 | int64, float64, object 등 |
| `df['col'].unique()` | 고유값 목록 | 범주형 값 확인 |
| `df['col'].value_counts()` | 범주별 빈도 | 가장 많은 값 파악 |
| `df['col'].nunique()` | 고유값 개수 | 범주 수 확인 |

In [ ]:
print('=== shape ===')
print(mpg.shape)

print('\n=== head(3) ===')
print(mpg.head(3))

print('\n=== info ===')
mpg.info()

In [ ]:
print('=== describe ===')
print(mpg.describe())

print('\n=== origin 고유값 ===')
print(mpg['origin'].unique())
print('\n=== origin 빈도 ===')
print(mpg['origin'].value_counts())

## 3-2. 대표값 (Representative Values)

데이터의 **중심 경향**을 나타내는 값으로, 분포의 중심을 파악합니다.

| 대표값 | 설명 | 함수 | 언제 사용? |
|--------|------|------|------------|
| 평균 (Mean) | 모든 값의 합 ÷ 개수 | `df['col'].mean()` | 이상치 없는 연속형 데이터 |
| 중앙값 (Median) | 정렬 후 가운데 값 | `df['col'].median()` | 이상치 있거나 왜곡 분포 |
| 최빈값 (Mode) | 가장 자주 나오는 값 | `df['col'].mode()[0]` | 범주형·이산형 데이터 |

> **핵심**: 평균과 중앙값 차이가 크면 → 이상치나 왜곡 분포를 의심!

In [ ]:
print('=== mpg 연비 대표값 ===')
print(f'평균(mean):    {mpg["mpg"].mean():.2f}')
print(f'중앙값(median):{mpg["mpg"].median():.2f}')
print(f'최빈값(mode):  {mpg["mpg"].mode()[0]:.2f}')

print('\n=== penguins 몸무게 대표값 ===')
print(f'평균:   {penguins["body_mass_g"].mean():.2f}')
print(f'중앙값: {penguins["body_mass_g"].median():.2f}')

## 3-3. 결측치와 이상치

### 결측치 (Missing Values)

**결측치**: 데이터에 값이 존재하지 않는 경우 (NaN, None)

| 작업 | 코드 | 설명 |
|------|------|------|
| 결측치 수 확인 | `df.isnull().sum()` | 열별 결측치 수 |
| 결측치 비율 | `df.isnull().mean() * 100` | 비율(%) |
| 행 제거 | `df.dropna()` | 결측치 있는 행 삭제 |
| 열 제거 | `df.dropna(axis=1)` | 결측치 있는 열 삭제 |
| 평균으로 대체 | `df.fillna(df.mean())` | 수치형 |
| 중앙값으로 대체 | `df.fillna(df.median())` | 이상치 있을 때 권장 |
| 최빈값으로 대체 | `df.fillna(df.mode().iloc[0])` | 범주형 |
| 앞값으로 채우기 | `df.fillna(method='ffill')` | 시계열 |
| 특정값으로 대체 | `df.fillna(0)` | 상황에 맞게 설정 |

In [ ]:
print('=== 결측치 수 ===')
print(penguins.isnull().sum())

print('\n=== 결측치 비율(%) ===')
print((penguins.isnull().mean() * 100).round(2))

# 수치형: 중앙값으로 채우기
penguins_filled = penguins.copy()
num_cols = penguins_filled.select_dtypes(include='number').columns
penguins_filled[num_cols] = penguins_filled[num_cols].fillna(
    penguins_filled[num_cols].median()
)
print('\n처리 후 결측치 수:', penguins_filled.isnull().sum().sum())

### 이상치 (Outliers)

**이상치**: 다른 값들에 비해 극단적으로 크거나 작은 값

**탐지 방법:**

| 방법 | 기준 | 특징 |
|------|------|------|
| IQR 방법 | `Q1 - 1.5×IQR` ~ `Q3 + 1.5×IQR` 범위 밖 | 가장 일반적, 박스플롯 기반 |
| Z-score | `|Z| ≥ 2` 또는 `|Z| ≥ 3` | 정규분포 가정 필요 |
| 박스플롯 | 수염(whisker) 밖의 점 | 시각적 탐지 |

**처리 방법:**

| 방법 | 코드/설명 |
|------|----------|
| 제거 | 이상치 행 `df.drop()` |
| 대체 | 중앙값 또는 경계값으로 교체 |
| 변환 | 로그/제곱근 변환 (`np.log1p()`) |
| 보존 | 플래그 컬럼 추가 후 유지 |

In [ ]:
# IQR 방법으로 이상치 탐지
col = 'body_mass_g'
data = penguins[col].dropna()

Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = data[(data < lower) | (data > upper)]
print(f'하한: {lower:.0f}, 상한: {upper:.0f}')
print(f'이상치 수: {len(outliers)}')

# 박스플롯으로 확인
sns.boxplot(x=penguins[col])
plt.title('Penguin Body Mass - Outlier Detection')
plt.show()

## 3-4. 산포도 (Spread / Dispersion)

데이터가 **얼마나 넓게 퍼져 있는지** 나타내는 지표입니다.
평균이 같아도 산포도가 다르면 전혀 다른 특성을 가집니다.

| 지표 | 계산 | 함수 | 장점 | 단점 |
|------|------|------|------|------|
| 범위 (Range) | 최댓값 - 최솟값 | `max() - min()` | 간단·직관적 | 이상치에 민감 |
| 분산 (Variance) | 편차 제곱의 평균 | `df.var()` | 모든 값 반영 | 단위가 달라짐 (cm²) |
| 표준편차 (Std) | √분산 | `df.std()` | 원래 단위 유지 | 이상치에 민감 |
| IQR | Q3 - Q1 | `Q3 - Q1` | 이상치에 강건 | 극단값 무시 |

> **해석**: std가 클수록 데이터가 평균에서 멀리 퍼져 있음

In [ ]:
cols = ['mpg', 'horsepower', 'weight']
for col in cols:
    data = mpg[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    print(f'=== {col} ===')
    print(f'  범위(Range):    {data.max() - data.min():.2f}')
    print(f'  분산(Variance): {data.var():.2f}')
    print(f'  표준편차(Std):  {data.std():.2f}')
    print(f'  IQR:            {Q3 - Q1:.2f}')

## 3-5. 사분위수 (Quartiles)

데이터를 오름차순 정렬 후 **4등분**하는 경계값입니다.

| 사분위수 | 백분위 | 의미 |
|----------|--------|------|
| Q1 (1사분위) | 25th percentile | 하위 25% 경계값 |
| Q2 (2사분위) | 50th percentile | 중앙값 (Median) |
| Q3 (3사분위) | 75th percentile | 하위 75% 경계값 |
| IQR | Q3 - Q1 | 중간 50% 데이터 범위 |

**IQR 활용:**
- 이상치 경계: `Q1 - 1.5×IQR` (하한) ~ `Q3 + 1.5×IQR` (상한)
- Q2 위치로 데이터 편향(skewness) 파악
- 박스플롯(Box Plot)의 상자 = Q1 ~ Q3 범위

In [ ]:
print('=== mpg 연비 사분위수 ===')
q_stats = mpg['mpg'].quantile([0.25, 0.5, 0.75])
print(q_stats)

Q1 = mpg['mpg'].quantile(0.25)
Q3 = mpg['mpg'].quantile(0.75)
IQR = Q3 - Q1
print(f'\nIQR: {IQR:.2f}')
print(f'이상치 하한: {Q1 - 1.5*IQR:.2f}')
print(f'이상치 상한: {Q3 + 1.5*IQR:.2f}')

# 박스플롯으로 사분위수 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=mpg['mpg'], ax=axes[0])
axes[0].set_title('mpg 연비 분포')
sns.boxplot(y=penguins['body_mass_g'], ax=axes[1])
axes[1].set_title('penguins 몸무게 분포')
plt.tight_layout()
plt.show()

## 3-6. 데이터 변형 (Pivot & Melt)

### Long Format vs Wide Format

| 형태 | 특징 | 적합한 상황 |
|------|------|-------------|
| **Wide Format** | 열이 많음, 한 행에 여러 측정값 | 스프레드시트, 비교 가독성 |
| **Long Format** | 행이 많음, 변수명+값 열 분리 | 시각화(Seaborn), ML 모델링 |

| 함수 | 방향 | 주요 파라미터 |
|------|------|---------------|
| `pivot_table()` | Long → Wide | `index`, `columns`, `values`, `aggfunc` |
| `melt()` | Wide → Long | `id_vars`, `value_vars`, `var_name`, `value_name` |

```python
# Pivot: Long → Wide
df.pivot_table(index='행기준', columns='열기준', values='값', aggfunc='mean')

# Melt: Wide → Long
df.melt(id_vars=['고정컬럼'], value_vars=['변환컬럼들'],
        var_name='변수명', value_name='값명')
```

In [ ]:
# 예시 데이터 (Long format)
data_long = pd.DataFrame({
    '지역': ['서울', '부산', '서울', '부산'],
    '연도': [2023, 2023, 2024, 2024],
    '판매량': [100, 80, 120, 90]
})
print('=== Long Format ===')
print(data_long)

# Pivot: Long → Wide
pivot_df = data_long.pivot_table(index='지역', columns='연도', values='판매량')
print('\n=== Pivot 결과 (Wide Format) ===')
print(pivot_df)

In [ ]:
# 예시 데이터 (Wide format)
data_wide = pd.DataFrame({
    '지역': ['서울', '부산'],
    '2023': [100, 80],
    '2024': [120, 90]
})
print('=== Wide Format ===')
print(data_wide)

# Melt: Wide → Long
melted = data_wide.melt(
    id_vars=['지역'],
    var_name='연도',
    value_name='판매량'
)
print('\n=== Melt 결과 (Long Format) ===')
print(melted)

## 실습 문제

### 문제
아래 조건에 맞는 DataFrame을 파이썬으로 생성하고, CSV 파일로 저장하는 코드를 작성하시오.

**조건:**
- 10명의 고객의 3개월(2025.1.1 ~ 2025.3.31) 구매 데이터
- 컬럼: `고객ID`, `월`, `상품카테고리`, `구매금액`
- 행 수: 30행 이상
- 상품카테고리: 음식, 의류, 전자제품
- 구매금액: 10,000원 ~ 100,000원 사이

````{admonition} 풀이
:class: dropdown

```python
import pandas as pd
import numpy as np

np.random.seed(42)

customers = [f'C{i:03d}' for i in range(1, 11)]  # C001 ~ C010
months = ['2025-01', '2025-02', '2025-03']
categories = ['음식', '의류', '전자제품']

rows = []
for cust in customers:
    for month in months:
        cat = np.random.choice(categories)
        price = round(np.random.randint(10000, 100001) / 10) * 10
        rows.append({
            '고객ID': cust,
            '월': month,
            '상품카테고리': cat,
            '구매금액': price
        })

df = pd.DataFrame(rows)
print(f'행 수: {len(df)}')
print(df.head(10))

df.to_csv('purchase_data.csv', index=False, encoding='utf-8-sig')
print('저장 완료: purchase_data.csv')
```
````

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

customers = [f'C{i:03d}' for i in range(1, 11)]
months = ['2025-01', '2025-02', '2025-03']
categories = ['음식', '의류', '전자제품']

rows = []
for cust in customers:
    for month in months:
        cat = np.random.choice(categories)
        price = round(np.random.randint(10000, 100001) / 10) * 10
        rows.append({'고객ID': cust, '월': month, '상품카테고리': cat, '구매금액': price})

df = pd.DataFrame(rows)
print(f'행 수: {len(df)}')
print(df.head(10))
print('\n구매금액 통계:')
print(df['구매금액'].describe())